# 03b - Drone Custom Environment

## Block 4: Your First Custom Environment

Up to this point, we have relied on pre-built environments provided by Gymnasium. However, in the real world, you will almost always need to build your own custom environments to model your specific business problems, robots, or games.

For this final challenge, you will choose **one of the two custom environments below** to implement and solve.

**Your Mission:**
1. Read the specifications for both environments.
2. Choose *one* that interests you.
3. Analyze its `observation_space` and `action_space`.
4. Based on whether the spaces are discrete or continuous, decide which algorithm we covered today (**Q-Learning** or **REINFORCE**) is the correct tool for the job.
5. Train an agent to solve it!

---

### Option B: The Hovering Drone (`DroneHoverEnv`)



In this environment, your agent is the flight controller for a drone. Instead of discrete buttons, it must smoothly adjust its vertical thrust to hover exactly at a specific target altitude.

| Feature | Description |
| :--- | :--- |
| **Observation Space** | `Box(1,)`: A single continuous float representing the current altitude. |
| **Action Space** | `Box(1,)`: A continuous thrust value ranging from `-1.0` (thrust down) to `1.0` (thrust up). |
| **Goal** | Maintain a `target_altitude` of 10.0. |
| **Rewards** | `-error` (the absolute difference between the current altitude and the target altitude). |
| **Termination / Truncation** | Episode terminates if the drone flies completely out of bounds (altitude < -10.0 or > 30.0), or truncates after `50` steps. |

---

### Challenge Instructions

Below, you will find the raw Python code defining the Drone environment.

1. Run the cell to register the class in your notebook.
2. Fill in the missing `TODO`s to complete the environment setup and physics.
3. Instantiate it using `env = DroneHoverEnv()`
4. Build the training loop! *Hint: Look back at Block 2 and Block 3. Which framework aligns with continuous action spaces?*

In [ ]:
from __future__ import annotations

import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from torch.distributions.normal import Normal

import gymnasium as gym


plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DroneHoverEnv(gym.Env):
    """
    A trivial 1D continuous environment.
    State: Altitude (float)
    Action: Vertical Thrust [-1.0, 1.0]
    Goal: Hover at target_altitude (10.0)
    """
    metadata = {"render_modes": ["human"]}

    def __init__(self, render_mode=None):
        super().__init__()
        self.render_mode = render_mode
        self.max_steps = 50

        # TODO: Observation: Current altitude (1D float). Hint: use spaces.Box
        self.observation_space = ...

        # TODO: Action: Thrust [-1.0 (down) to 1.0 (up)]
        self.action_space = ...

        self.target_altitude = 10.0
        self.current_step = 0
        self.altitude = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0

        # TODO: Start at a random altitude between 0 and 20
        self.altitude = ...

        return np.array([self.altitude], dtype=np.float32), {}

    def step(self, action):
        self.current_step += 1

        # Trivial Physics: Thrust directly changes altitude
        thrust = action[0]
        # TODO: change the current altitude
        self.altitude += ...

        # TODO: Trivial Reward: Distance from target
        error = ...
        reward = -error # BE CAREFUL: The negative sign is already here!

        # Terminate if it flies completely out of bounds (optional fail-safe)
        terminated = bool(self.altitude < -10.0 or self.altitude > 30.0)
        truncated = self.current_step >= self.max_steps

        return np.array([self.altitude], dtype=np.float32), float(reward), terminated, truncated, {}

    def render(self):
        if self.render_mode != "human":
            return

        # A simple text-based renderer
        # Represents altitude from 0 to 20
        alt_int = int(np.clip(self.altitude, 0, 20))
        target_int = int(self.target_altitude)

        track = ["-"] * 21
        track[target_int] = "T" # Target

        if 0 <= alt_int <= 20:
            if alt_int == target_int:
                track[alt_int] = "🎯" # Perfectly on target
            else:
                track[alt_int] = "🚁" # Drone

        print(f"Step {self.current_step:02d} | {''.join(track)} | Alt: {self.altitude:.2f}")

    def close(self):
        pass

In [ ]:
import torch
import random
import numpy as np

# Create and wrap the environment
# TODO: create the env
env = ...
LR=1e-4
wrapped_env = gym.wrappers.RecordEpisodeStatistics(env, 50)  # Records episode-reward

total_num_episodes = int(3e3)  # Total number of episodes
obs_space_dims = env.observation_space.shape[0]
action_space_dims = env.action_space.shape[0]

rewards_over_seeds = []

for seed in [1, 2, 3]:  # You can add more seeds here later to average performance
    # Set global seeds
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

    # Reinitialize agent every seed
    # TODO: initialize the agent
    agent = ...
    reward_over_episodes = []

    for episode in range(total_num_episodes):

        # FIXED: Only apply the seed on the very first episode!
        if episode == 0:
            obs, info = wrapped_env.reset(seed=seed)
        else:
            obs, info = wrapped_env.reset()

        done = False
        while not done:
            # TODO: Sample action
            action = ...

            ... = wrapped_env.step(action)


            agent.rewards.append(reward)

            done = terminated or truncated

        # RecordEpisodeStatistics stores the total episode return here
        reward_over_episodes.append(wrapped_env.return_queue[-1])

        # TODO: we need the agent to update
        agent ...


        # Print progress
        if episode % 1000 == 0:
            avg_reward = float(np.mean(wrapped_env.return_queue))
            print(f"Episode: {episode} | Average Reward (last 50): {avg_reward:.2f}")

    rewards_over_seeds.append(reward_over_episodes)

In [ ]:
# TODO: Run the cell below to test your agent (if errors occur check for name of classes/methods)

In [ ]:
# @title
import numpy as np

def test_continuous_agent(agent, env, num_episodes=100, render_last=True):
    """Test continuous agent performance without learning."""
    total_rewards = []
    successes = 0

    for i in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done = False

        # Decide if we want to print the text render for this specific episode
        should_render = render_last and (i == num_episodes - 1)

        if should_render:
            print(f"\n--- Rendering Final Test Episode ({env.__class__.__name__}) ---")
            env.render()

        while not done:
            # For testing, you ideally want the deterministic action (the mean of the policy)
            action = agent.sample_action(obs)

            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated

            if should_render:
                env.render()

        # Check if the episode was a success:
        # Survived without flying out of bounds AND finished close to the target
        final_altitude = obs[0]
        close_enough = abs(final_altitude - env.target_altitude) <= 1.0

        if truncated and not terminated and close_enough:
            successes += 1

        total_rewards.append(episode_reward)

    win_rate = successes / num_episodes
    average_reward = np.mean(total_rewards)

    print(f"\nTest Results over {num_episodes} episodes:")
    print(f"Hover Success Rate (Within 1m of target): {win_rate:.1%}")
    print(f"Average Reward: {average_reward:.3f}")
    print(f"Standard Deviation: {np.std(total_rewards):.3f}")


# Test your agent
# Assuming DroneHoverEnv was defined in your notebook
env = DroneHoverEnv(render_mode="human")
test_continuous_agent(agent, env, num_episodes=100, render_last=True)